# Dupin · Fase 2 — Construcción de la matriz de features

Reproduce el stream de PaySim en orden temporal y construye la matriz de features
de comportamiento `feat-v1`, publicándola a `gs://dupin-dupin-features/feat-v1/`.

**Paridad por construcción:** este notebook **clona el repo** y usa el mismo
`features/build_features.py` que usará el serving. No reimplementa nada — es
físicamente imposible que entrenamiento y producción difieran (Invariante 1).

Decisiones (Fase 1): comportamiento sobre `nameDest`, superficie {TRANSFER,
CASH_OUT}, ventanas 24h/168h + histórico + recencia. Columnas de balance
EXCLUIDAS (fuga de etiqueta).

## 1. Clonar el repo (mismo `features/` que el serving)

In [ ]:
from google.colab import userdata
import sys, subprocess

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")   # PAT con acceso de lectura al repo privado
REPO = "alexxcode/dupin"

subprocess.run(["rm", "-rf", "/content/dupin"])
subprocess.run(["git", "clone", f"https://{GITHUB_TOKEN}@github.com/{REPO}.git", "/content/dupin"], check=True)
if "/content/dupin" not in sys.path:
    sys.path.insert(0, "/content/dupin")
print("Repo clonado. HEAD:")
subprocess.run(["git", "-C", "/content/dupin", "log", "--oneline", "-1"])

In [ ]:
!pip -q install gcsfs pyarrow

## 2. Configuración y autenticación GCP

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import storage

PROJECT_ID   = "dupin-dupin"
BUCKET_RAW   = "dupin-dupin-raw"
BUCKET_FEAT  = "dupin-dupin-features"
REGION       = "us-central1"
RAW_OBJECT   = "raw/paysim/PS_20174392719_1491204439457_log.csv"
GCS_RAW      = f"gs://{BUCKET_RAW}/{RAW_OBJECT}"

# Crea el bucket de features si no existe.
client = storage.Client(project=PROJECT_ID)
if not client.bucket(BUCKET_FEAT).exists():
    client.create_bucket(BUCKET_FEAT, location=REGION)
    print("Bucket creado:", BUCKET_FEAT)
else:
    print("Bucket ya existe:", BUCKET_FEAT)

## 3. Cargar raw (solo columnas necesarias)

No cargamos las 4 columnas de balance: no son features (fuga) y ahorran memoria.

In [ ]:
import pandas as pd

USE_COLS = ["step", "type", "amount", "nameOrig", "nameDest", "isFraud"]
DTYPES = {"step":"int32","type":"category","amount":"float64",
          "nameOrig":"string","nameDest":"string","isFraud":"int8"}

df = pd.read_csv(GCS_RAW, usecols=USE_COLS, dtype=DTYPES,
                 storage_options={"project": PROJECT_ID})
print(f"Filas: {len(df):,}   Columnas: {list(df.columns)}")
df.head(3)

## 4. Construir features (paridad por construcción)

In [ ]:
import time
from features.build_features import build_features
from features.config import DEFAULT_CONFIG, FEATURE_NAMES, META_COLUMNS

print("Construyendo... (reproduce 6.36M tx en orden temporal; tarda unos minutos)")
t0 = time.time()
matrix = build_features(df, DEFAULT_CONFIG, with_meta=True)
dt = time.time() - t0
print(f"Listo en {dt/60:.1f} min. Matriz: {matrix.shape}")
matrix.head(3)

## 5. Sanity checks

In [ ]:
n = len(matrix)
nf = int(matrix["isFraud"].sum())
print(f"Filas emitidas (superficie TRANSFER+CASH_OUT): {n:,}")
print(f"Fraude: {nf:,}  ({nf/n:.4%})   <- tasa base sube vs 0.13% global")
print(f"Tipos: {sorted(matrix['type'].unique().tolist())}")

# Estructura: columnas exactas, sin NaN, sin columnas de balance (fuga).
assert list(matrix.columns) == META_COLUMNS + FEATURE_NAMES, "columnas inesperadas"
assert matrix[FEATURE_NAMES].isna().sum().sum() == 0, "hay NaN en features"
for banned in ["oldbalanceOrg","newbalanceOrig","oldbalanceDest","newbalanceDest"]:
    assert banned not in matrix.columns, f"columna con fuga presente: {banned}"
print("\nOK: columnas correctas, sin NaN, sin columnas de balance.")

display(matrix[FEATURE_NAMES].describe().T)

## 6. Publicar a GCS (`features/feat-v1/`)

In [ ]:
import json, datetime

FEAT_PREFIX = f"gs://{BUCKET_FEAT}/{DEFAULT_CONFIG.feature_version}"
OUT_PARQUET = f"{FEAT_PREFIX}/features.parquet"

matrix.to_parquet(OUT_PARQUET, index=False, engine="pyarrow",
                  storage_options={"project": PROJECT_ID})
print("Matriz escrita:", OUT_PARQUET)

# Metadata versionada junto a la matriz.
meta = {
    "feature_version": DEFAULT_CONFIG.feature_version,
    "source_raw": GCS_RAW,
    "n_rows": int(n),
    "n_fraud": int(nf),
    "fraud_rate": nf / n,
    "scope_types": list(DEFAULT_CONFIG.scope_types),
    "windows": DEFAULT_CONFIG.windows,
    "feature_names": FEATURE_NAMES,
    "created_at": datetime.datetime.utcnow().isoformat() + "Z",
}
blob = client.bucket(BUCKET_FEAT).blob(f"{DEFAULT_CONFIG.feature_version}/metadata.json")
blob.upload_from_string(json.dumps(meta, indent=2), content_type="application/json")
print("Metadata escrita:", f"{FEAT_PREFIX}/metadata.json")
print(json.dumps(meta, indent=2))

---
**Fase 2 completa** cuando este notebook corre limpio. Salida:
`gs://dupin-dupin-features/feat-v1/features.parquet` + `metadata.json`.

La matriz alimenta la Fase 3 (split temporal + evaluación honesta) y la Fase 4
(entrenamiento). El serving (Fase 5) usará exactamente este `features/`.